<a href="https://colab.research.google.com/github/rulogar1234/nuclio-1/blob/develop/SUP_ML_3v1_Predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SUP ML 3 - PREDICT

# Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pickle
from sklearn.pipeline import Pipeline,FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn import set_config
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder


pd.set_option('display.max_columns',None)

import warnings
warnings.filterwarnings("ignore")

# Carga modelo

* Cargar el modelo
* Obtener la lista de model features

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#para que las salidas sean dataframes
set_config(transform_output = "pandas")

In [ ]:
PATH_DATA='/content/drive/MyDrive/Colab Notebooks/TEMPLATE SUP ML (main)/EJERCICIO-ML-SUP/data/'
modelo_pickle_=PATH_DATA+'modelo_pickle.pkl'
model = pickle.load(open(modelo_pickle_, 'rb'))

# Carga PREDICT dataset

In [ ]:
ruta_predict_='/content/drive/MyDrive/Colab Notebooks/TEMPLATE SUP ML (main)/EJERCICIO-ML-SUP/data/telecom_churn_PREDICT.csv'
df=pd.read_csv(ruta_predict_)
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.head()

# ML Preprocessing

Se deben realizar las mismas transformaciones que se aplicaron en el preprocessing del train:
  * Eliminar mismas variables
  * Imputar mismos valores a nulos
  * Aplicar mismo encoding de categoricos
  * ...todo lo necesario para replicar las variables con las que el modelo se entren?

NO SE DEBEN repetir:
  * Analisis visual de los datos
  * Analisis de correlaciones
  * Analisis de varianza
  * Calculo de metricas que varien con la distribuci?n

In [ ]:
df.set_index('Customer_ID',inplace=True)

In [ ]:
target = 'churn'
features = df.columns[df.columns!=target]

# Divide dataset en matriz de variables (X) y output (y)
X = df[features]

In [ ]:
# Check duplicados
df.index.nunique() != len(df.index)

In [ ]:
def check_data_types(df):
    for col_name, col_type in df.dtypes.items():
        col_data = df[col_name]
        if col_type == np.float64 and np.issubdtype(col_data.dtype, np.integer):
            print(f"La columna {col_name} es de tipo float64 pero solo contiene datos de tipo int64")
            return 1
        elif col_type == np.int64 and np.issubdtype(col_data.dtype, np.floating):
            print(f"La columna {col_name} es de tipo int64 pero solo contiene datos de tipo float64")
            return 1
        elif col_type == np.object_ and np.issubdtype(col_data.dtype, np.datetime64):
            print(f"La columna {col_name} es de tipo object pero solo contiene datos de tipo datetime64")
            return 1
        elif col_type == np.datetime64 and np.issubdtype(col_data.dtype, np.object_):
            print(f"La columna {col_name} es de tipo datetime64 pero solo contiene datos de tipo object")
            return 1
        elif col_type != col_data.dtype:
            print(f"La columna {col_name} es de tipo {col_type} pero contiene datos de tipo {col_data.dtype}")
            return 1
    return 0

In [ ]:
check_data_types(df)

In [ ]:
#Borramos los mismos campos
df.drop('infobase', axis=1, inplace=True)
df.drop('HHstatin', axis=1, inplace=True)
df.drop('truck', axis=1, inplace=True)
df.drop('models', axis=1, inplace=True)
df.drop('rv', axis=1, inplace=True)
df.drop('lor', axis=1, inplace=True)
df.drop('dwlltype', axis=1, inplace=True)
df.drop('numbcars', axis=1, inplace=True)
df.drop('forgntvl', axis=1, inplace=True)
df.drop('ownrent', axis=1, inplace=True)
df.drop('adults', axis=1, inplace=True)
df.drop('dwllsize', axis=1, inplace=True)
df.drop('ethnic', axis=1, inplace=True)
df.drop('avg6mou', axis=1, inplace=True)
df.drop('avg6qty', axis=1, inplace=True)
df.drop('avg6rev', axis=1, inplace=True)
df.drop('dualband', axis=1, inplace=True)

In [ ]:
df2=df.copy()

In [ ]:
def columnas_con_nulos(dataframe):
    columnas_nulas = dataframe.columns[dataframe.isnull().any()].tolist()
    return columnas_nulas
def crear_dataframe_nulos(dataframe):
    columnas_nulas = columnas_con_nulos(dataframe)
    df_nulos = dataframe[columnas_nulas].copy()
    return df_nulos

# Crear el DataFrame con las columnas que tienen valores nulos y sus valores correspondientes para poder trabajar con undataset m?s peque?o
df_nulos = crear_dataframe_nulos(df2)

In [ ]:
df_nulos.shape

In [ ]:
df_nulos.isnull().sum()

In [ ]:
valores_clasif_num=df_nulos.select_dtypes(include=['float64','int64']).columns.tolist()
print(valores_clasif_num)

In [ ]:
#Vamos a quitar hnd_price y a plicar la misma media que en train
valores_clasif_num.remove('hnd_price')
media_hnd_price = 99.70957487428218

In [ ]:
#creamos una lista con los nombres de las columnas clasificatorias con nulos
indice_clasif=df_nulos.select_dtypes(include='object').columns.tolist()
indice_clasif

In [ ]:
#Hacemos una funci?n para separar las variables que deben tener U y las que deben tener Unknown y las que deber?n tratarse con un ordinal (kids)
valores_clasif_1=[]
valores_clasif_mas=[]
valores_clasif_kid=[]
for i in indice_clasif:
  l=df_nulos[i].str.len().max()
  kid=i.startswith("kid")
  if l == 1 and kid==False:
    valores_clasif_1.append(i)
  elif l == 1 and kid==True:
    valores_clasif_kid.append(i)
  else:
    valores_clasif_mas.append(i)

print(valores_clasif_1)

In [ ]:
#a?adimos el area todo y que no tenga nulos para igualar el nombre
valores_clasif_mas.append('area')
print(valores_clasif_mas)

In [ ]:
print(valores_clasif_kid)

In [ ]:
#Funciones
def income_outlayers_assign(X):
    X.loc[X['income'].isnull(), 'income'] = outlayer_num
    X.loc[X['income'].notnull(), 'income'] = X['income']
    return X

def eqpdays_outlayers_assign(X):
    X.loc[X['eqpdays'] < 0, 'eqpdays'] = outlayer_num
    X.loc[X['eqpdays'] >= 0, 'eqpdays'] = X['eqpdays']
    return X

#creamos una funci?n para hacer tratamiento ordinal del campo tipo kids
def kids0_2_outlayers_assign(X):
  X.loc[X['kid0_2'] == 'Y','kid0_2'] = 1
  X.loc[X['kid0_2'] == 'U','kid0_2'] = 0
  X.loc[X['kid0_2'].isnull(),'kid0_2'] = X['kid0_2']
  return X

def kids3_5_outlayers_assign(X):
  X.loc[X['kid3_5'] == 'Y','kid3_5'] = 1
  X.loc[X['kid3_5'] == 'U','kid3_5'] = 0
  X.loc[X['kid3_5'].isnull(),'kid3_5'] = X['kid3_5']
  return X

def kids6_10_outlayers_assign(X):
  X.loc[X['kid6_10'] == 'Y','kid6_10'] = 1
  X.loc[X['kid6_10'] == 'U','kid6_10'] = 0
  X.loc[X['kid6_10'].isnull(),'kid6_10'] = X['kid6_10']
  return X

def kids11_15_outlayers_assign(X):
  X.loc[X['kid11_15'] == 'Y','kid11_15'] = 1
  X.loc[X['kid11_15'] == 'U','kid11_15'] = 0
  X.loc[X['kid11_15'].isnull(),'kid11_15'] = X['kid11_15']
  return X

def kids16_17_outlayers_assign(X):
  X.loc[X['kid16_17'] == 'Y','kid16_17'] = 1
  X.loc[X['kid16_17'] == 'U','kid16_17'] = 0
  X.loc[X['kid16_17'].isnull(),'kid16_17'] = X['kid16_17']
  return X




In [ ]:
#asignamos el mismo outlayer num?rico que en train
outlayer_num=-3875000

In [ ]:
impute = ColumnTransformer(
    transformers=[
        ("col_1",SimpleImputer(strategy="constant",fill_value='-U'),valores_clasif_1),
        ("col_mas",SimpleImputer(strategy="constant",fill_value='-Unknown'),valores_clasif_mas),
        ("col_num",SimpleImputer(strategy="constant",fill_value=outlayer_num),valores_clasif_num),
        ("hnd_price",SimpleImputer(strategy="constant",fill_value=media_hnd_price),["hnd_price"]),
        ("income", FunctionTransformer(func=income_outlayers_assign),["income"]),
        ("eqpdays",FunctionTransformer(func=eqpdays_outlayers_assign),['eqpdays']),
        ("kid_1",FunctionTransformer(func=kids0_2_outlayers_assign),['kid0_2']),
        ("kid_2",FunctionTransformer(func=kids3_5_outlayers_assign),['kid3_5']),
        ("kid_3",FunctionTransformer(func=kids6_10_outlayers_assign),['kid6_10']),
        ("kid_4",FunctionTransformer(func=kids11_15_outlayers_assign),['kid11_15']),
        ("kid_5",FunctionTransformer(func=kids16_17_outlayers_assign),['kid16_17'])

    ],
    remainder='passthrough'
)

In [ ]:
df_t=impute.fit_transform(df2)

In [ ]:
def prizm_social_one_unkw(X):
    X.loc[X['col_1__prizm_social_one'] == '-U', 'col_1__prizm_social_one'] = 'U'
    X.loc[X['col_1__prizm_social_one'] != '-U', 'col_1__prizm_social_one'] = X['col_1__prizm_social_one']
    return X

def hnd_webcap_unkw(X):
    X.loc[X['col_mas__hnd_webcap'] == '-Unknown', 'col_mas__hnd_webcap'] = 'UNKW'
    X.loc[X['col_mas__hnd_webcap'] != '-Unknown', 'col_mas__hnd_webcap'] = X['col_mas__hnd_webcap']
    return X

def marital_unkw(X):
    X.loc[X['col_1__marital'] == '-U', 'col_1__marital'] = 'U'
    X.loc[X['col_1__marital'] != '-U', 'col_1__marital'] = X['col_1__marital']
    return X

#se ha procedido a tratar cada columan separadamente
def kid_assign(X):
  resultado =  X['kid_1__kid0_2']+X['kid_2__kid3_5']+X['kid_3__kid6_10']+X['kid_4__kid11_15']+X['kid_5__kid16_17']
  return pd.DataFrame(resultado, columns=['kids'])

In [ ]:
impute2 = ColumnTransformer(
    transformers=[
         ("kids",FunctionTransformer(func=kid_assign),['kid_1__kid0_2','kid_2__kid3_5','kid_3__kid6_10','kid_4__kid11_15','kid_5__kid16_17']),
         ("crclscod",OrdinalEncoder(),['remainder__crclscod']),
         ("asl_flag",OrdinalEncoder(),['remainder__asl_flag']),
         ("refurb_new",OrdinalEncoder(),['remainder__refurb_new']),
         ("prizm_social_one",FunctionTransformer(func=prizm_social_one_unkw),['col_1__prizm_social_one']),
         ("hnd_webcap",FunctionTransformer(func=hnd_webcap_unkw),['col_mas__hnd_webcap']),
         ("marital",FunctionTransformer(func=marital_unkw),['col_1__marital']),
         ("area",OneHotEncoder(sparse_output=False),["col_mas__area"]),
         ("new_cell",OneHotEncoder(sparse_output=False),["remainder__new_cell"]),
         ("creditcd",OneHotEncoder(sparse_output=False),["col_1__creditcd"])
    ],
    remainder='passthrough'
)


In [ ]:
df_t2=impute2.fit_transform(df_t)

In [ ]:
impute3 = ColumnTransformer(
    transformers = [
        ("hnd_web_cap",OneHotEncoder(sparse_output=False),["hnd_webcap__col_mas__hnd_webcap"]),
        ("prizm_social_one",OneHotEncoder(sparse_output=False),["prizm_social_one__col_1__prizm_social_one"]),
        ("marital",OneHotEncoder(sparse_output=False),["marital__col_1__marital"]),
        ("kids",SimpleImputer(strategy="constant",fill_value=outlayer_num),["kids__kids"])
    ],
    remainder="passthrough"

)

In [ ]:
df_t3=impute3.fit_transform(df_t2)

In [ ]:
#creamos la func?n para transformar las dos columnas que son numeros(col_kid__kids y dwllsize__col_1__dwllsize) a integers y la web_cap usaremos ahora s? un OneHotEncoder
def cambiar_columna_int(column):
    return column.astype('int64')

In [ ]:
#pasamos kids a integer
impute4 = ColumnTransformer(
    transformers = [
        ("kids",FunctionTransformer(func=cambiar_columna_int),['kids__kids__kids'])
    ],
    remainder="passthrough"

)

In [ ]:
df_t4=impute4.fit_transform(df_t3)

In [ ]:
pipe = Pipeline(
    steps = [
        ("imputer1",impute),
        ("imputer2",impute2),
        ("imputer3",impute3),
        ("imputer4",impute4)
    ]
)

In [ ]:
df_t4=pipe.fit_transform(df2)

In [ ]:
df_t4.info()

In [ ]:
def ConvertFloatsToInt(X):
  lista_columnas=[]
  for i in X:
      if X[i].dtype.kind =="f" and np.modf(X[i])[0].sum()==0 and X[i].isnull().sum()==0:
        lista_columnas.append([i])
        X[i]=X[i].astype('int64')
  print(lista_columnas)
  return X

In [ ]:
df_t5 = ConvertFloatsToInt(df_t4)

In [ ]:
df_t5.columns.to_list()

In [ ]:
df_t5.shape

In [ ]:
aborrar_=  ['remainder__remainder__creditcd__col_1__creditcd_-U', 'remainder__remainder__creditcd__col_1__creditcd_Y', 'remainder__remainder__remainder__col_num__rev', 'remainder__remainder__remainder__col_num__totmrc', 'remainder__remainder__remainder__col_num__da', 'remainder__remainder__remainder__col_num__ovrmou', 'remainder__remainder__remainder__col_num__ovrrev', 'remainder__remainder__remainder__col_num__vceovr', 'remainder__remainder__remainder__col_num__datovr', 'remainder__remainder__remainder__col_num__roam', 'remainder__remainder__remainder__col_num__change_rev', 'remainder__remainder__remainder__col_num__income', 'remainder__remainder__remainder__remainder__plcd_vce', 'remainder__remainder__remainder__remainder__comp_dat', 'remainder__remainder__remainder__remainder__cc_mou', 'remainder__remainder__remainder__remainder__inonemin', 'remainder__remainder__remainder__remainder__mou_opkd', 'remainder__remainder__remainder__remainder__attempt', 'remainder__remainder__remainder__remainder__comp_vce', 'remainder__remainder__remainder__remainder__totrev', 'remainder__remainder__remainder__remainder__totmou', 'remainder__remainder__remainder__remainder__totcalls']

In [ ]:
df_t5.drop(aborrar_, axis=1, inplace=True)

In [ ]:
df_t5.shape

In [ ]:
df_t5.info()

In [ ]:
df_t5.shape

# Check model features

* Comprobar que tenemos en el dataset preprocesado todas las model features, de lo contrario no podremos hacer predict.
* Ordenar las variables en mismo orden que las model features

In [ ]:
#orden de las variables
columnas = df_t5.columns
for i in columnas:
  print(i)

# Rescaling

* Si se entren? el modelo con un dataset estandarizado, estandarizar con mismo scaler.

# PREDICT

* predict() y predict_proba()

In [ ]:
# Check features matched features model
features_predict = list(df_t5.columns)
model_features = list(model.feature_names_in_)

In [ ]:
features_predict == model_features

In [ ]:
predict = model.predict(df_t5)

In [ ]:
predict

In [ ]:
#probabilidad
predict_proba = model.predict_proba(df_t5)[:,1]

In [ ]:
predict_proba

# Guarda predicciones

* Guardar las predicciones en data path. Cada fila debe estar etiquetada con el ID.

In [ ]:
pd.DataFrame(predict).to_csv(PATH_DATA+'churn_prediction.csv', index=True)